# Integração entre Dados das API's do IBGE e BC

Este notebook integrará os dados recebidos pelas API's do IBGE e do BC e irá uni-los em uma só tabela

## Imports

In [38]:
import json
import pandas as pd
import requests as rqs
from pathlib import Path
import altair as alt
import geobr
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from ipywidgets import Dropdown, Layout, interactive_output

## Reunir os arquivos da API do BC

Os dados da API do BC estão na pasta dados > bcb > cache em formato tipo .json

In [2]:
dados_da_api_bc = []

# Pesquisa todos os arquivos .json em dados > bcb > cache
for arquivo in Path("dados/bcb/cache").glob("*.json"):
    with open(arquivo, encoding="utf-8") as f:
        cache = json.load(f)
    dados_da_api_bc.append(pd.DataFrame(cache["dados"]["value"]))

sicor = pd.concat(dados_da_api_bc, ignore_index=True)

## Rodar o código da API_IBGE

Rodando o código com o %run e colocando o %%capture para não dar print em nada na tela, ou seja, apenas rodar e carregar todas as tabelas do API_IBGE

In [3]:
%%capture
%run API_IBGE.ipynb

## Padronizando os dados da API_BC

Padronizando os dados da API_BC para igualar os valores dos dados aos da API_IBGE e tratar a tabela

In [4]:
# Renomeando as colunas com o mesmo nome das colunas da API_IBGE
sicor = sicor.rename(columns = {'nomeProduto':'cultura', 'AnoEmissao':'ano', 'MesEmissao':'mes', 'Municipio':'municipio' })

# Padronizando o nome da cultura
sicor['cultura'] = sicor['cultura'].replace({
    '"SOJA"':'Soja (em grão)',
    '"CANA-DE-ALÚCAR"' :'Cana-de-açúcar',
    '"MILHO"':'Milho (em grão)'
})

# Deixando o municipio em letra minúscula
sicor['municipio'] = sicor['municipio'].str.title()
sicor
# Renomenado as colunas relacionadas com Crédito rural
sicor = sicor.rename(columns={
    'cdFonteRecurso': 'fonte de recurso',
    'cdPrograma': 'programa de financiamento',
    'cdSubPrograma': 'subprograma',
    'cdModalidade': 'modalidade',
    'Atividade': 'atividade',
    'cdTipoSeguro': 'tipo de seguro',
    'VlCusteio': 'valor de custeio',
    'AreaCusteio': 'area de custeio',
})

#cria a tabela com os dados úteis do BC
tabela_bc = sicor.drop(columns=['codCadMu', 'cdProduto', 'codIbge','cdEstado'])

## Padronizando os dados da API_IBGE

A padronização será adicionar todos os dados municipais em uma única tabela

In [5]:
#Unindo todas as tabelas com base nas colunas iguais
tabela_ibge_municipal = df_area_municipal.merge(df_prod_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])
tabela_ibge_municipal = tabela_ibge_municipal.merge(df_rendimento_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])

#Renomendo as colunas e retirando as colunas redundantes
tabela_ibge_municipal = tabela_ibge_municipal.drop(columns=['variavel_x','unidade_x','variavel_y','unidade_y', 'variavel','unidade'])
tabela_ibge_municipal = tabela_ibge_municipal.rename(columns={
    'valor_x':'area plantada (hectares)',
    'valor_y':'quantidade produzida (toneladas)',
    'valor':'rendimento medio (kg por hectare)',
    'codigo_cultura':'codigo cultura',
    'codigo_municipio':'codigo municipio'
})

# Adicionando a sigla do estado
tabela_ibge_municipal['estado'] = tabela_ibge_municipal['municipio'].str.split('-').str[1].str.strip()

# Padronizando o municipio para ficar idêntico ao da API_BC
tabela_ibge_municipal['municipio'] = tabela_ibge_municipal['municipio'].str.split("-").str[0].str.strip().str.title().astype(str)

# Ordenar
nova_ordem = [
    'ano',
    'estado',
    'municipio',
    'codigo municipio',
    'cultura',
    'codigo cultura',
    'area plantada (hectares)',
    'quantidade produzida (toneladas)',
    'rendimento medio (kg por hectare)'
]
tabela_ibge_municipal = tabela_ibge_municipal[nova_ordem]
tabela_ibge_municipal

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare)
0,2016,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0
1,2017,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,5.0,300.0,60000.0
2,2018,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,4.0,238.0,59500.0
3,2019,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,15.0,1208.0,80533.0
4,2020,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
164758,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0
164759,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0
164760,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0
164761,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0


## Dicionário para padronização da 2° tabela (a seguir)

In [6]:
programas = {
    "0001": "PRONAF",
    "0050": "PRONAMP",
    "0999": "Sem vínculo a programa específico",
    "0070": "Funcafé",
    "0151": "Procap-Agro",
    "0152": "Proirriga (antigo Moderinfra)",
    "0153": "Moderagro",
    "0154": "Moderfrota",
    "0155": "Prodecoop",
    "0162": "Inovagro",
    "0163": "PCA (armazéns)",
    "0201": "Crédito Fundiário (FTRA)",
    "0222": "Renovagro",
    "0240": "ANF (atividade não financiada, Proagro)",
}

fontes = {
    "0100": "TESOURO NACIONAL - DIRECIONADA/CONTROLADA",
    "0201": "OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA",
    "0202": "EXIGIBILIDADE ADICIONAL DOS RECURSOS À VISTA",
    "0222": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5030 ENCERRADO",
    "0225": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5087 - ENCERRADO",
    "0226": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5157",
    "0250": "FACULDADE DE APLICAÇÃO - COMPULSÓRIO",
    "0260": "COMPULSÓRIO SOBRE RECURSOS À VISTA - REFORÇO DO INVESTIMENTO (CIRC 3.745)",
    "0300": "POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0301": "POUPANÇA RURAL - CONTROLADOS - CONDIÇÕES MCR 6.2",
    "0302": "POUPANÇA RURAL - CONTROLADOS - FATOR DE PONDERAÇÃO",
    "0303": "POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA",
    "0304": "EXIGIBILIDADE ADICIONAL DA POUPANÇA RURAL",
    "0402": "RECURSOS LIVRES - LIVRE/NÃO CONTROLADA",
    "0403": "RECURSOS LIVRES - EQUALIZADA - LIVRE/CONTROLADA",
    "0405": "FUNDO DE COMMODITIES",
    "0430": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECIONADA/NÃO CONTROLADA",
    "0431": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0440": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - TAXA FAVORECIDA",
    "0450": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - EQUALIZÁVEL",
    "0451": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - DIRECIONADA",
    "0501": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORTE (FNO) - DIRECIONADA/CONTROLADA",
    "0502": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORDESTE (FNE) - DIRECIONADA/CONTROLADA",
    "0503": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO CENTRO-OESTE (FCO) - DIRECIONADA/CONTROLADA",
    "0505": "BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0506": "BNDES - LIVRE/NÃO CONTROLADA",
    "0507": "INCRA - DIRECIONADA/CONTROLADA",
    "0520": "FUNDO DE TERRAS E DA REFORMA AGRÁRIA - DIRECIONADA/CONTROLADA",
    "0600": "GOVERNOS E FUNDOS ESTADUAIS OU MUNICIPAIS - DIRECIONADA/CONTROLADA",
    "0650": "FAT - FUNDO DE AMPARO AO TRABALHADOR - DIRECIONADA/CONTROLADA",
    "0680": "PIS/PASEP",
    "0800": "FUNCAFE - FUNDO DE DEFESA DA ECONOMIA CAFEEIRA - DIRECIONADA/CONTROLADA",
    "0850": "CAPTAÇÃO EXTERNA - LIVRE/NÃO CONTROLADA",
    "0900": "ATIVIDADE NÃO FINANCIADA ENQUADRADA NO PROAGRO (MCR 16.8)",
    "0910": "FUNDO NACIONAL SOBRE MUDANÇA (FUNDO CLIMA) - RES CMN 5.130/2024",
    "0911": "FUNDO DE DESENVOLVIMENTO CIENTÍFICO E TECNOLÓGICO (FNDCT) - MP 1374 DIRECIONADA/CONTROLADA",
    "0990": "OUTRAS FONTES DE RECURSOS NÃO ESPECIFICADAS",
}

tipo_seguro = {
    0: "Não se aplica",
    1: "Proagro tradicional",
    2: "Proagro mais",
    3: "Outro seguro",
    9: "Sem adesão a seguro",
}

meses = {
    "01": "Janeiro",
    "02": "Fevereiro",
    "03": "Março",
    "04": "Abril",
    "05": "Maio",
    "06": "Junho",
    "07": "Julho",
    "08": "Agosto",
    "09": "Setembro",
    "10": "Outubro",
    "11": "Novembro",
    "12": "Dezembro"
}


## Criação de duas tabelas com os dados do BC

1° Tabela: tabela agrupada com o valor do crédito rural por municipio, cultura e ano

2° Tabela: tabela padronizada com os nomes de cada coluna corretamente

In [7]:
# 1° Tabela: tabela do crédito rural
colunas_a_serem_somadas = ['valor de custeio', 'area de custeio']
tabela_credito_rural = sicor.groupby(['municipio','cultura','ano'])[colunas_a_serem_somadas].sum().reset_index()
# Transformando o ano em int
tabela_credito_rural['ano'] = tabela_credito_rural['ano'].astype(int)

# 2° Tabela: tabela com os nomes padronizados com o dicionário de cima
tabela_padronizada = sicor.copy()
tabela_padronizada['programa de financiamento'] = sicor['programa de financiamento'].replace(programas)
tabela_padronizada['fonte de recurso'] = sicor['fonte de recurso'].replace(fontes)
tabela_padronizada['tipo de seguro'] = sicor['tipo de seguro'].replace(tipo_seguro)
# ANALISAR SE PRECISA PADRONIZAR O MÊS
# tabela_padronizada['mes'] = sicor['mes'].replace(meses)
tabela_padronizada['mes'] = sicor['mes'].astype(int)

tabela_padronizada


,municipio,cultura,mes,ano,programa de financiamento,subprograma,fonte de recurso,tipo de seguro,cdEstado,valor de custeio,cdProduto,codCadMu,atividade,modalidade,codIbge,area de custeio
0,Jesuítas,Soja (em grão),7,2021,Sem vínculo a programa específico,0,POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA,9,19,120515.25,6720,40785,1,01,4112751,40.29
1,Flor Da Serra Do Sul,Soja (em grão),7,2021,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,3,19,884242.51,6720,49982,1,01,4107850,264.96
2,Dianópolis,Soja (em grão),7,2021,Sem vínculo a programa específico,0,LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECI...,3,28,354260.00,6720,12036,1,01,1707009,100.00
3,Nova Araçá,Soja (em grão),7,2021,PRONAMP,0,BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTRO...,1,24,44311.34,6720,34832,1,01,4312807,12.00
4,Iporã,Soja (em grão),7,2021,PRONAMP,0,BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTRO...,3,19,129270.72,6720,30953,1,01,4110607,42.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831985,Caseara,Soja (em grão),1,2018,Sem vínculo a programa específico,0,POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA,9,28,1150000.00,6720,43751,1,01,1703909,564.89
831986,Presidente Bernardes,Soja (em grão),1,2018,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,2,27,27267.81,6720,40163,1,01,3541208,16.94
831987,Nobres,Soja (em grão),1,2018,Sem vínculo a programa específico,0,OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA,3,14,2606343.93,6720,12988,1,01,5105903,1278.70
831988,Águas Frias,Soja (em grão),1,2018,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,2,25,5892.00,6720,49494,1,01,4200556,4.00


## Unir as tabelas

Irá unir-se as tabelas: tabela_ibge_municipal com tabela_credito_rural. Entretanto, há alguns valores que estão apenas em uma das duas tabelas, logo esses valores possuirão NaN e serão removidos.

In [8]:
tabela_unida = tabela_ibge_municipal.merge(tabela_credito_rural, on=['municipio', 'ano','cultura'])
tabela_unida = tabela_unida.dropna()
tabela_unida

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare),valor de custeio,area de custeio
0,2021,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,0.0,0.0,0.0,800000.00,1.55
1,2022,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,18.0,1450.0,80556.0,800000.00,1.35
2,2023,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,15.0,1275.0,85000.0,1500000.00,1.74
3,2024,GO,Abadia De Goiás,5200050,Cana-de-açúcar,2696,13.0,350.0,29167.0,700000.00,1.11
4,2021,GO,Abadia De Goiás,5200050,Milho (em grão),2711,200.0,1360.0,6800.0,1220142.65,463.97
...,...,...,...,...,...,...,...,...,...,...,...
57281,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0,3529324.92,1043.23
57282,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0,7584973.10,1292.92
57283,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0,8117759.43,1607.08
57284,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0,6289436.44,1176.02


## Base validada para comparar municipios de areas semelhantes

A funcao `validar_tabela()` usa a base integrada `dados/tabela_final.csv` e retorna uma nova tabela com cobertura `ambos`, area plantada positiva e credito numerico, finito e nao negativo. Credito zero e aceito; credito ausente nao vira zero. As exclusoes sao informadas por etapa, sem dupla contagem.

`tabela_validada` e salva em `dados/tabela_validada.csv`, preservando a base completa. Esta tabela apoia a pergunta 4. A quantidade de contratos e mantida quando disponivel e fica ausente quando a fonte nao a fornece; contar linhas nao substitui contratos. Os caches municipais atuais nao incluem esse campo.


In [ ]:
def validar_tabela(tabela):
    """Filtra cobertura ambos, area positiva e credito finito nao negativo."""
    validada = tabela.copy()
    total = len(validada)

    anterior = len(validada)
    validada = validada.loc[validada["cobertura"].eq("ambos")].copy()
    print(f"Excluidos por cobertura: {anterior - len(validada):,}")

    area = pd.to_numeric(validada["area_plantada_ha"], errors="coerce")
    anterior = len(validada)
    validada["area_plantada_ha"] = area
    validada = validada.loc[(area.gt(0) & area.lt(float("inf"))).fillna(False)].copy()
    print(f"Excluidos por area ausente, invalida ou nao positiva: {anterior - len(validada):,}")

    credito = pd.to_numeric(validada["credito_nominal_brl"], errors="coerce")
    anterior = len(validada)
    validada["credito_nominal_brl"] = credito
    validada = validada.loc[(credito.ge(0) & credito.lt(float("inf"))).fillna(False)].copy()
    print(f"Excluidos por credito ausente, invalido ou negativo: {anterior - len(validada):,}")

    if "quantidade_contratos" not in validada.columns:
        validada["quantidade_contratos"] = pd.Series(pd.NA, index=validada.index, dtype="Int64")
    print(f"Total: {total:,} | Excluidos: {total - len(validada):,} | Mantidos: {len(validada):,}")
    print(f"Registros sem quantidade de contratos: {validada['quantidade_contratos'].isna().sum():,}")
    return validada.reset_index(drop=True)


tabela_completa = pd.read_csv("dados/tabela_final.csv", dtype={"codigo_municipio": "string"})
tabela_validada = validar_tabela(tabela_completa)
tabela_validada.to_csv("dados/tabela_validada.csv", index=False)
tabela_validada.head()


Excluidos por cobertura: 110,788
Excluidos por area ausente, invalida ou nao positiva: 1,335
Excluidos por credito ausente, invalido ou negativo: 0
Total: 164,775 | Excluidos: 112,123 | Mantidos: 52,652
Registros sem quantidade de contratos: 52,652


## Faixas relativas de area plantada

`categorizar_faixas_hectares(tabela, amplitude=0.25)` recebe a tabela validada e classifica cada registro municipio-cultura-ano. Dentro de cada cultura e ano, os limites comecam na menor area observada e crescem 25% a cada faixa. Nao ha um corte unico de 200 hectares nem um numero fixo de grupos. A amplitude e uma escolha metodologica ajustavel, nao um limite estabelecido pelo IBGE.

O limite inferior pertence ao intervalo e o superior pertence ao seguinte: `[100; 125)` inclui 100 hectares e exclui 125. Os rotulos sao arredondados para leitura; `area_min_faixa_ha` e `area_max_faixa_ha` preservam os limites numericos utilizados. `faixa_area_ordem` permite ordenar os grupos no grafico.

A entrada deve ter area positiva e finita, como a saida de `validar_tabela()`. A funcao acrescenta colunas sem alterar as areas originais. O resultado fica em `tabela_categorizada` e em `dados/tabela_categorizada.csv`; `resumo_faixas` informa quantos municipios ha em cada faixa. Grupos pequenos devem ser identificados antes de calcular estatisticas.

Esta classificacao apoia a pergunta 4: comparar o credito de municipios com areas proximas, na mesma cultura e ano. Uma faixa de numero igual em culturas ou anos diferentes pode ter limites distintos; compare sempre os hectares explicitos. O recorte de cobertura continua sendo o da tabela validada e a quantidade de contratos ausente continua ausente.


In [ ]:
def categorizar_faixas_hectares(tabela, amplitude=0.25):
    """Categoriza a tabela validada em faixas relativas por cultura e ano."""
    fator = 1 + amplitude
    if not 1 < fator < float("inf"):
        raise ValueError("A amplitude deve ser finita e maior que zero.")

    partes = []
    for _, grupo in tabela.groupby(["cultura", "ano"], sort=False):
        grupo = grupo.copy()
        area = grupo["area_plantada_ha"]
        limites = [float(area.min())]
        while limites[-1] <= area.max():
            limites.append(limites[-1] * fator)

        faixa = pd.cut(area, bins=limites, labels=False, right=False)
        grupo["faixa_area_ordem"] = faixa + 1
        grupo["area_min_faixa_ha"] = faixa.map(dict(enumerate(limites[:-1])))
        grupo["area_max_faixa_ha"] = faixa.map(dict(enumerate(limites[1:])))
        grupo["faixa_hectares"] = (
            "[" + grupo["area_min_faixa_ha"].round(2).astype(str)
            + "; " + grupo["area_max_faixa_ha"].round(2).astype(str) + ") ha"
        )
        partes.append(grupo)

    if not partes:
        return tabela.assign(
            faixa_area_ordem=pd.Series(dtype="int64"),
            area_min_faixa_ha=pd.Series(dtype="float64"),
            area_max_faixa_ha=pd.Series(dtype="float64"),
            faixa_hectares=pd.Series(dtype="string"),
        )
    return pd.concat(partes).sort_index()


tabela_categorizada = categorizar_faixas_hectares(tabela_validada)
tabela_categorizada.to_csv("dados/tabela_categorizada.csv", index=False)
resumo_faixas = (
    tabela_categorizada.groupby(
        ["cultura", "ano", "faixa_area_ordem", "faixa_hectares"], as_index=False
    ).agg(municipios=("codigo_municipio", "nunique"))
)
print(f"Registros categorizados: {len(tabela_categorizada):,}")
print(f"Faixas ocupadas por cultura e ano: {len(resumo_faixas):,}")
print(f"Faixas com apenas um municipio: {resumo_faixas['municipios'].eq(1).sum():,}")
tabela_categorizada.head()


Registros categorizados: 52,652
Faixas ocupadas por cultura e ano: 1,458
Faixas com apenas um municipio: 123


## Estatisticas por cultura, ano e faixa de area

`resumir_estatisticas_faixas(tabela_categorizada)` retorna `tabela_estatistica`, com uma linha por cultura, ano e faixa. Calcula numero de municipios distintos, mediana da area, mediana do credito, quartis Q1 (25%) e Q3 (75%) e intervalo interquartil (IQR = Q3 - Q1).

O credito por hectare e calculado primeiro em cada municipio, dividindo credito por area plantada. A funcao resume a distribuicao desses valores com mediana, quartis e IQR; nao divide a mediana do credito pela mediana da area. Os valores de credito sao nominais, em reais do respectivo ano, e os indicadores de intensidade estao em R$/ha.

Quando ha contagens municipais de contratos validas (inteiras, nao negativas e finitas), inclui mediana, quartis e IQR dos contratos e dos contratos por mil hectares. Tambem calcula a mediana dos valores medios municipais por contrato. `municipios_com_contratos` informa a cobertura dessas medidas. A divisao por contrato considera apenas contagens positivas; zero e ausencia nao viram denominadores. Nos grupos sem contagens, as estatisticas permanecem ausentes.

O CSV e salvo em `dados/tabela_estatistica_faixas.csv`, sem modificar a tabela categorizada. A base atual nao tem contagens de contratos, portanto so disponibiliza as estatisticas de area e credito. Grupos com um municipio sao mantidos e identificados: IQR zero nesses grupos nao demonstra homogeneidade entre municipios. Esta tabela apoia a pergunta 4 e deve ser examinada antes dos graficos.


In [ ]:
def resumir_estatisticas_faixas(tabela):
    """Resume area, credito e, quando disponiveis, contratos por faixa."""
    base = tabela.copy()
    chaves = ["cultura", "ano", "faixa_area_ordem", "faixa_hectares"]
    base["credito_por_hectare_brl"] = base["credito_nominal_brl"] / base["area_plantada_ha"]
    estatisticas = {
        "area_min_faixa_ha": ("area_min_faixa_ha", "first"),
        "area_max_faixa_ha": ("area_max_faixa_ha", "first"),
        "numero_municipios": ("codigo_municipio", "nunique"),
        "mediana_area_ha": ("area_plantada_ha", "median"),
        "mediana_credito_brl": ("credito_nominal_brl", "median"),
        "q1_credito_brl": ("credito_nominal_brl", lambda s: s.quantile(0.25)),
        "q3_credito_brl": ("credito_nominal_brl", lambda s: s.quantile(0.75)),
        "mediana_credito_por_hectare_brl": ("credito_por_hectare_brl", "median"),
        "q1_credito_por_hectare_brl": ("credito_por_hectare_brl", lambda s: s.quantile(0.25)),
        "q3_credito_por_hectare_brl": ("credito_por_hectare_brl", lambda s: s.quantile(0.75)),
    }

    contratos_disponiveis = False
    if "quantidade_contratos" in base.columns:
        contratos = pd.to_numeric(base["quantidade_contratos"], errors="coerce")
        base["quantidade_contratos"] = contratos.where(
            contratos.ge(0) & contratos.lt(float("inf")) & contratos.mod(1).eq(0)
        )
        estatisticas["municipios_com_contratos"] = ("quantidade_contratos", "count")
        contratos_disponiveis = base["quantidade_contratos"].notna().any()
        if contratos_disponiveis:
            base["contratos_por_mil_hectares"] = base["quantidade_contratos"] * 1000 / base["area_plantada_ha"]
            base["valor_medio_por_contrato_brl"] = base["credito_nominal_brl"] / base["quantidade_contratos"].where(base["quantidade_contratos"].gt(0))
            estatisticas.update({
                "mediana_contratos": ("quantidade_contratos", "median"),
                "q1_contratos": ("quantidade_contratos", lambda s: s.quantile(0.25)),
                "q3_contratos": ("quantidade_contratos", lambda s: s.quantile(0.75)),
                "mediana_contratos_por_mil_hectares": ("contratos_por_mil_hectares", "median"),
                "q1_contratos_por_mil_hectares": ("contratos_por_mil_hectares", lambda s: s.quantile(0.25)),
                "q3_contratos_por_mil_hectares": ("contratos_por_mil_hectares", lambda s: s.quantile(0.75)),
                "mediana_valor_medio_por_contrato_brl": ("valor_medio_por_contrato_brl", "median"),
                "municipios_com_valor_por_contrato": ("valor_medio_por_contrato_brl", "count"),
            })

    resumo = base.groupby(chaves, as_index=False).agg(**estatisticas)
    resumo["iqr_credito_brl"] = resumo["q3_credito_brl"] - resumo["q1_credito_brl"]
    resumo["iqr_credito_por_hectare_brl"] = resumo["q3_credito_por_hectare_brl"] - resumo["q1_credito_por_hectare_brl"]
    if contratos_disponiveis:
        resumo["iqr_contratos"] = resumo["q3_contratos"] - resumo["q1_contratos"]
        resumo["iqr_contratos_por_mil_hectares"] = resumo["q3_contratos_por_mil_hectares"] - resumo["q1_contratos_por_mil_hectares"]
    return resumo.sort_values(["cultura", "ano", "faixa_area_ordem"]).reset_index(drop=True)


tabela_estatistica = resumir_estatisticas_faixas(tabela_categorizada)
tabela_estatistica.to_csv("dados/tabela_estatistica_faixas.csv", index=False)
print(f"Grupos resumidos: {len(tabela_estatistica):,}")
print(f"Grupos com apenas um municipio: {tabela_estatistica['numero_municipios'].eq(1).sum():,}")
if "mediana_contratos" not in tabela_estatistica.columns:
    print("Sem contagens validas de contratos na base atual; estatisticas de contratos nao calculadas.")
display(tabela_estatistica.head(10))


Grupos resumidos: 1,458
Grupos com apenas um municipio: 123
Sem contagens validas de contratos na base atual; estatisticas de contratos nao calculadas.


cultura,ano,faixa_area_ordem,faixa_hectares,area_min_faixa_ha,area_max_faixa_ha,numero_municipios,mediana_area_ha,mediana_credito_brl,q1_credito_brl,q3_credito_brl,mediana_credito_por_hectare_brl,q1_credito_por_hectare_brl,q3_credito_por_hectare_brl,municipios_com_contratos,iqr_credito_brl,iqr_credito_por_hectare_brl
cana_de_acucar,2016,1,[5.0; 6.25) ha,5.000000,6.250000,1,5.0,14323.300,14323.3000,1.432330e+04,2864.660000,2864.660000,2.864660e+03,0,0.0000,0.000000
cana_de_acucar,2016,2,[6.25; 7.81) ha,6.250000,7.812500,2,7.0,93927.840,50167.8450,1.376878e+05,13418.262857,7166.835000,1.966969e+04,0,87519.9900,12502.855714
cana_de_acucar,2016,4,[9.77; 12.21) ha,9.765625,12.207031,2,10.0,6753275.295,3379912.9425,1.012664e+07,675327.529500,337991.294250,1.012664e+06,0,6746724.7050,674672.470500
cana_de_acucar,2016,6,[15.26; 19.07) ha,15.258789,19.073486,2,17.5,307500.000,170250.0000,4.447500e+05,17137.254902,9539.215686,2.473529e+04,0,274500.0000,15196.078431
cana_de_acucar,2016,7,[19.07; 23.84) ha,19.073486,23.841858,4,20.0,27906.130,4108.6600,9.763704e+04,1395.306500,205.433000,4.495466e+03,0,93528.3775,4290.032946
cana_de_acucar,2016,8,[23.84; 29.8) ha,23.841858,29.802322,3,25.0,5000.000,4080.1300,2.025000e+05,200.000000,163.205200,8.100000e+03,0,198419.8700,7936.794800
cana_de_acucar,2016,9,[29.8; 37.25) ha,29.802322,37.252903,5,30.0,22472.020,7118.9600,3.248862e+04,702.250625,203.398857,1.082954e+03,0,25369.6600,879.555143
cana_de_acucar,2016,10,[37.25; 46.57) ha,37.252903,46.566129,4,40.0,7919.140,5027.7625,1.104602e+04,197.978500,126.691832,2.683397e+02,0,6018.2600,141.647889
cana_de_acucar,2016,11,[46.57; 58.21) ha,46.566129,58.207661,4,50.0,55000.000,20499.6600,9.976417e+04,1100.000000,409.993200,1.981160e+03,0,79264.5100,1571.166996
cana_de_acucar,2016,12,[58.21; 72.76) ha,58.207661,72.759576,6,67.0,7479.215,4215.2325,4.345000e+04,105.624018,65.094188,7.189286e+02,0,39234.7675,653.834384


## Gráficos

## Dispersao: credito e area plantada por municipio

O grafico utiliza `tabela_categorizada`, com os registros individuais de cobertura `ambos`, area positiva e credito valido. Cada ponto representa um municipio na cultura e no ano selecionados. O eixo X mostra hectares; o eixo Y, credito de custeio nominal em reais. As cores identificam regioes.

Selecione cultura, ano e, opcionalmente, UF. Passe o mouse para consultar municipio, codigo IBGE, valores e faixa de hectares. Use a roda do mouse para ampliar e arraste para navegar; duplo clique restaura o enquadramento. As escalas sao lineares. Para a pergunta 4, observe a dispersao vertical entre pontos proximos no eixo da area: ela indica diferentes valores de credito em municipios com areas semelhantes.

Os controles e o zoom seguem o [exemplo oficial do Altair](https://altair-viz.github.io/gallery/multiple_interactions.html). O grafico e exibido diretamente no notebook, sem criar pasta nem exportar arquivos. A classificacao e as tabelas existentes sao preservadas. A comparacao descreve associacoes, nao suas causas; nao inclui quantidade de contratos, ausente na fonte municipal atual.


In [ ]:
def criar_grafico_dispersao_area_credito(tabela):
    """Cria dispersao municipal com filtros de cultura, ano e UF."""
    colunas = [
        "codigo_municipio", "municipio", "uf", "regiao", "cultura", "ano",
        "area_plantada_ha", "credito_nominal_brl", "faixa_hectares",
    ]
    base = tabela[colunas].copy()
    culturas = sorted(base["cultura"].unique().tolist())
    anos = sorted(base["ano"].unique().tolist())
    ufs = ["Todas"] + sorted(base["uf"].unique().tolist())
    cultura = alt.param(
        name="cultura_dispersao", value="soja" if "soja" in culturas else culturas[0],
        bind=alt.binding_select(options=culturas, name="Cultura: "),
    )
    ano = alt.param(
        name="ano_dispersao", value=anos[-1],
        bind=alt.binding_select(options=anos, name="Ano: "),
    )
    uf = alt.param(
        name="uf_dispersao", value="Todas",
        bind=alt.binding_select(options=ufs, name="UF: "),
    )
    # InlineData permite usar a base completa sem mudar o limite global do Altair.
    return (
        alt.Chart(alt.InlineData(values=base.to_dict(orient="records")))
        .mark_circle(size=45, opacity=0.5, clip=True)
        .add_params(cultura, ano, uf)
        .transform_filter(alt.datum.cultura == cultura)
        .transform_filter(alt.datum.ano == ano)
        .transform_filter((uf == "Todas") | (alt.datum.uf == uf))
        .encode(
            x=alt.X("area_plantada_ha:Q", title="Area plantada (ha)", axis=alt.Axis(format=",.0f")),
            y=alt.Y("credito_nominal_brl:Q", title="Credito de custeio (R$ nominais)", axis=alt.Axis(format=",.0f")),
            color=alt.Color("regiao:N", title="Regiao"),
            tooltip=[
                alt.Tooltip("municipio:N", title="Municipio"),
                alt.Tooltip("codigo_municipio:N", title="Codigo IBGE"),
                alt.Tooltip("uf:N", title="UF"),
                alt.Tooltip("cultura:N", title="Cultura"),
                alt.Tooltip("ano:O", title="Ano"),
                alt.Tooltip("area_plantada_ha:Q", title="Area (ha)", format=",.2f"),
                alt.Tooltip("credito_nominal_brl:Q", title="Credito (R$)", format=",.2f"),
                alt.Tooltip("faixa_hectares:N", title="Faixa de area"),
            ],
        )
        .properties(
            width=850, height=450,
            title=alt.Title(
                "Credito entre municipios com areas plantadas semelhantes",
                subtitle="Um ponto por municipio no ano e cultura selecionados; cor por regiao.",
            ),
        )
        .interactive()
    )


grafico_dispersao_area_credito = criar_grafico_dispersao_area_credito(tabela_categorizada)
grafico_dispersao_area_credito


## Questao 5 - Credito por tonelada e por hectare entre estados

Os dois graficos comparam estados dentro de uma cultura e ano. O indicador estadual e a razao dos totais: soma do credito dividida pela soma da producao (R$/t) ou da area plantada (R$/ha). Essa divisao ja relativiza o credito pelo tamanho agricola; nao se calcula a media simples das razoes municipais nem uma segunda divisao pelo denominador.

A entrada e `tabela_validada`, de cobertura `ambos`, area positiva e credito valido. Para R$/t, municipios com producao ausente, nao finita ou nao positiva sao retirados tanto do numerador quanto do denominador. Para R$/ha, permanece o recorte de area positiva. A celula informa as exclusoes de cada indicador; por isso, os dois paineis podem representar conjuntos diferentes de municipios.

**Cobertura:** os valores sao agregados estaduais dos municipios cobertos pela base, nao necessariamente totais integrais de cada UF. O tooltip mostra quantidade de municipios, credito e denominador usados. A linha tracejada e a razao entre os totais nacionais dessa mesma cobertura, para a cultura e o ano selecionados; nao e uma media simples dos estados. Estados sem observacoes validas nao aparecem como credito zero.

Selecione cultura e ano e passe o mouse sobre as barras. As UFs sao ordenadas pelo indicador de cada painel, com escalas independentes porque as unidades diferem. A selecao inicial e soja no ano mais recente. Os graficos sao exibidos apenas no notebook, sem exportacao. Estes paineis respondem a parte estadual da pergunta 5; nao mostram a distribuicao municipal nem medem eficiencia ou custo de producao.


In [ ]:
def calcular_indicadores_estaduais(tabela):
    """Calcula razoes dos totais estaduais na cobertura da base validada."""
    partes = []
    for indicador, coluna, unidade in [
        ("credito_por_tonelada", "producao_t", "R$/t"),
        ("credito_por_hectare", "area_plantada_ha", "R$/ha"),
    ]:
        denominador = pd.to_numeric(tabela[coluna], errors="coerce")
        base = tabela.loc[(denominador.gt(0) & denominador.lt(float("inf"))).fillna(False)].copy()
        base[coluna] = denominador.loc[base.index]
        print(f"{unidade}: {len(tabela) - len(base):,} registros excluidos por denominador invalido ou nao positivo.")
        resumo = base.groupby(["cultura", "ano", "uf"], as_index=False).agg(
            credito_total_brl=("credito_nominal_brl", "sum"),
            denominador_total=(coluna, "sum"),
            numero_municipios=("codigo_municipio", "nunique"),
        )
        resumo["indicador"] = indicador
        resumo["unidade"] = unidade
        resumo["valor"] = resumo["credito_total_brl"] / resumo["denominador_total"]
        nacional = resumo.groupby(["cultura", "ano"])[["credito_total_brl", "denominador_total"]].transform("sum")
        resumo["referencia_nacional"] = nacional["credito_total_brl"] / nacional["denominador_total"]
        partes.append(resumo)
    return pd.concat(partes, ignore_index=True)


tabela_indicadores_estaduais = calcular_indicadores_estaduais(tabela_validada)
tabela_indicadores_estaduais.head()


R$/t: 46 registros excluidos por denominador invalido ou nao positivo.
R$/ha: 0 registros excluidos por denominador invalido ou nao positivo.


In [ ]:
def criar_graficos_estaduais_questao5(tabela):
    """Compara intensidades estaduais de credito, com filtros compartilhados."""
    culturas = sorted(tabela["cultura"].unique().tolist())
    anos = sorted(tabela["ano"].unique().tolist())
    cultura = alt.param(
        name="cultura_questao5", value="soja" if "soja" in culturas else culturas[0],
        bind=alt.binding_select(options=culturas, name="Cultura: "),
    )
    ano = alt.param(
        name="ano_questao5", value=anos[-1],
        bind=alt.binding_select(options=anos, name="Ano: "),
    )
    paineis = []
    for indicador, titulo, eixo, denominador, cor in [
        ("credito_por_tonelada", "Credito por tonelada", "Credito / producao (R$/t)", "Producao coberta (t)", "#2878a5"),
        ("credito_por_hectare", "Credito por hectare", "Credito / area plantada (R$/ha)", "Area coberta (ha)", "#2b8c68"),
    ]:
        base = (
            alt.Chart(tabela)
            .transform_filter(alt.datum.indicador == indicador)
            .transform_filter(alt.datum.cultura == cultura)
            .transform_filter(alt.datum.ano == ano)
        )
        barras = base.mark_bar(color=cor).encode(
            y=alt.Y("uf:N", sort="-x", title="UF"),
            x=alt.X("valor:Q", title=eixo, axis=alt.Axis(format=",.0f")),
            tooltip=[
                alt.Tooltip("uf:N", title="UF"),
                alt.Tooltip("cultura:N", title="Cultura"),
                alt.Tooltip("ano:O", title="Ano"),
                alt.Tooltip("valor:Q", title=eixo, format=",.2f"),
                alt.Tooltip("credito_total_brl:Q", title="Credito coberto (R$)", format=",.2f"),
                alt.Tooltip("denominador_total:Q", title=denominador, format=",.2f"),
                alt.Tooltip("numero_municipios:Q", title="Municipios cobertos", format=",.0f"),
                alt.Tooltip("referencia_nacional:Q", title="Referencia nacional", format=",.2f"),
            ],
        )
        referencia = (
            base.transform_aggregate(referencia_nacional="mean(referencia_nacional)")
            .mark_rule(color="#b64040", strokeDash=[5, 4], strokeWidth=2)
            .encode(
                x="referencia_nacional:Q",
                tooltip=[alt.Tooltip("referencia_nacional:Q", title="Referencia nacional", format=",.2f")],
            )
        )
        paineis.append((barras + referencia).properties(width=390, height=460, title=titulo))
    return (
        alt.hconcat(*paineis, spacing=50)
        .add_params(cultura, ano)
        .resolve_scale(y="independent", x="independent")
        .properties(title=alt.Title(
            "Questao 5 - Intensidade do credito entre estados",
            subtitle=[
                "Razao dos totais dos municipios com dados nas duas fontes; valores nominais do ano selecionado.",
                "Linha vermelha tracejada: referencia nacional calculada com a mesma cobertura de cada indicador.",
            ],
        ))
    )


graficos_estaduais_questao5 = criar_graficos_estaduais_questao5(tabela_indicadores_estaduais)
graficos_estaduais_questao5


Graficos estaduais interativos da questao 5.

### Gráfico 1) Como evoluíram o valor contratado e o número de contratos de custeio por cultura na década?

In [44]:
tabela_valor_contratado = sicor.groupby(['cultura', 'ano','mes'])['valor de custeio'].sum().reset_index()

# Coluna com a data formatada adequadamente
tabela_valor_contratado['data'] = pd.to_datetime({
    'year': tabela_valor_contratado['ano'],
    'month': tabela_valor_contratado['mes'],
    # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
    'day': 1
})

cores = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=['red','orange','green']
)

grafico_valor_contratado = alt.Chart(tabela_valor_contratado).mark_line().encode(
    x=alt.X('yearmonth(data):T', 
        title='Mês / Ano',
        axis=alt.Axis(format="%m/%Y", tickCount={"interval": "month", "step": 6})
    ),
    y=alt.Y('valor de custeio:Q',
        title='Valor de Custeio'
    ),
    color=alt.Color('cultura:N', scale=cores, title= 'Culturas')
).properties(
    title='Evolução do Valor de Custeio por Cultura',
    width=700,
    height=400
)
grafico_valor_contratado

alt.Chart(...)

In [43]:
tabela_numero_contratos = sicor.groupby(['cultura','ano','mes']).size().reset_index(name='quantidade')

# Coluna com a data formatada adequadamente
tabela_numero_contratos['data'] = pd.to_datetime({
    'year': tabela_numero_contratos['ano'],
    'month': tabela_numero_contratos['mes'],
    # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
    'day': 1
})

cores = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=['red','orange','green']
)

grafico_numero_contratos = alt.Chart(tabela_numero_contratos).mark_line().encode(
    x=alt.X('yearmonth(data):T', 
        title='Mês / Ano',
        axis=alt.Axis(format="%m/%Y", tickCount={"interval": "month", "step": 6})
    ),
    y=alt.Y('quantidade:Q',
        title='Quantidade de Contratos',
    ),
    color=alt.Color('cultura:N', scale=cores, title= 'Culturas')
).properties(
    title='Evolução da Quantidade de Contratos por Cultura',
    width=700,
    height=400
)

grafico_numero_contratos

alt.Chart(...)

### Gráfico 2) Como o crédito e a área plantada se distribuem entre estados e regiões?

In [65]:
# Comparar crédito nominal e área plantada nos anos em comum de cada cultura.
regioes = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}
area_g2 = df_area_estadual.rename(columns={"codigo_estado": "uf", "valor": "Área plantada (ha)"})
area_g2 = area_g2.assign(uf=area_g2["uf"].astype(str), ano=area_g2["ano"].astype(int))
credito_g2 = sicor.assign(
    uf=sicor["codIbge"].astype(str).str[:2],
    ano=sicor["ano"].astype(int),
).rename(columns={"valor de custeio": "Crédito de custeio (R$)"})

# Usar apenas os anos que aparecem nas duas fontes para cada cultura.
chave_ano = ["cultura", "ano"]
anos_comuns = area_g2[chave_ano].drop_duplicates().merge(
    credito_g2[chave_ano].drop_duplicates(), on=chave_ano
)
periodos = anos_comuns.groupby("cultura")["ano"].agg(["min", "max"])

indicadores_g2 = ["Crédito de custeio (R$)", "Área plantada (ha)"]
chave_uf = ["cultura", "uf"]
credito_comum = credito_g2.merge(anos_comuns, on=chave_ano)
area_comum = area_g2.merge(anos_comuns, on=chave_ano)

# Somar cada indicador por UF e juntar os resultados, incluindo UFs com dados em uma só fonte.
credito_por_uf = credito_comum.groupby(chave_uf)[indicadores_g2[0]].sum()
area_por_uf = area_comum.groupby(chave_uf)[indicadores_g2[1]].sum()
totais_g2 = pd.concat([credito_por_uf, area_por_uf], axis=1).fillna(0).reset_index()

nomes_uf = area_g2[["uf", "estado"]].drop_duplicates().set_index("uf")["estado"]
totais_g2["estado"] = totais_g2["uf"].map(nomes_uf).fillna("UF não identificada")
totais_g2["regiao"] = totais_g2["uf"].str[0].map(regioes).fillna("Região não identificada")

limite_estados = 8
cores_g2 = ["#2866A6", "#3A851D"]


def grafico_2(cultura):
    dados = totais_g2[totais_g2["cultura"] == cultura]
    por_regiao = dados.groupby("regiao")[indicadores_g2].sum()
    por_estado = dados.groupby("estado")[indicadores_g2].sum()

    # Ordenar pela área plantada e somar o restante em uma única categoria.
    por_estado = por_estado.sort_values("Área plantada (ha)", ascending=False, kind="stable")
    principais = por_estado.head(limite_estados).copy()
    demais = por_estado.iloc[limite_estados:]
    if not demais.empty:
        principais.loc["Outros"] = demais.sum()

    # Dividir cada indicador pela soma do Brasil.
    total_brasil = por_estado.sum().replace(0, float("nan"))
    participacao_regiao = por_regiao / total_brasil
    participacao_estado = principais / total_brasil


    fig, (ax_regiao, ax_estado) = plt.subplots(2, 1, figsize=(13.5, 7), layout="constrained")
    fig.get_layout_engine().set(hspace=0.25)
    participacao_regiao.plot.barh(ax=ax_regiao, color=cores_g2, legend=False, width=0.8)
    participacao_estado.plot.bar(ax=ax_estado, color=cores_g2, legend=False, rot=30, width=0.8)

    ax_regiao.set(title="Região", xlabel="Participação no total nacional", ylabel="")
    ax_regiao.title.set_fontsize(16)
    ax_estado.set(title="Estados", xlabel="", ylabel="Participação no total nacional")
    ax_estado.title.set_fontsize(16)
    ax_regiao.xaxis.set_major_formatter(PercentFormatter(1))
    ax_estado.yaxis.set_major_formatter(PercentFormatter(1))
    ax_regiao.legend(
        loc="lower center", bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False,
    )

    for eixo, direcao in [(ax_regiao, "x"), (ax_estado, "y")]:
        eixo.set_axisbelow(True)
        eixo.grid(axis=direcao, alpha=0.22)
        eixo.spines[["top", "right", "left"]].set_visible(False)
    for rotulo in ax_estado.get_xticklabels():
        rotulo.set_ha("right")

    primeiro_ano, ultimo_ano = periodos.loc[cultura]
    fig.suptitle(
        f"Distribuição do crédito de custeio e da área plantada\n"
        f"{cultura} · {primeiro_ano} a {ultimo_ano}",
        fontsize=18,
    )
    ax_regiao.tick_params(axis="y", labelsize=14)
    ax_regiao.tick_params(axis="x", labelsize=12)
    ax_estado.tick_params(axis="both", labelsize=12)
    plt.show()
    plt.close(fig)


seletor_g2 = Dropdown(
    options=periodos.index.tolist(), description="Cultura", layout=Layout(width="330px"),
)
saida_g2 = interactive_output(grafico_2, {"cultura": seletor_g2})
display(seletor_g2, saida_g2)

Dropdown(description='Cultura', layout=Layout(width='330px'), options=('Cana-de-açúcar', 'Milho (em grão)', 'S…

Output()

### Gráfico 3) O crédito está mais concentrado em poucos municípios do que a produção?

In [64]:
credito_g3 = sicor[["cultura", "ano", "codIbge", "valor de custeio"]].rename(
    columns={"codIbge": "municipio_id", "valor de custeio": "Crédito de custeio (R$)"}
)
producao_g3 = df_prod_municipal[["cultura", "ano", "codigo_municipio", "valor"]].rename(
    columns={"codigo_municipio": "municipio_id", "valor": "Produção (t)"}
)
credito_g3["municipio_id"] = credito_g3["municipio_id"].astype(str)
producao_g3["municipio_id"] = producao_g3["municipio_id"].astype(str)
credito_g3["ano"] = credito_g3["ano"].astype(int)
producao_g3["ano"] = producao_g3["ano"].astype(int)

# Comparar os dois indicadores nos mesmos anos para cada cultura.
chave_ano_g3 = ["cultura", "ano"]
anos_g3 = credito_g3[chave_ano_g3].drop_duplicates().merge(
    producao_g3[chave_ano_g3].drop_duplicates(), on=chave_ano_g3
)
periodos_g3 = anos_g3.groupby("cultura")["ano"].agg(["min", "max"])

chave_municipio_g3 = ["cultura", "municipio_id"]
coluna_credito = "Crédito de custeio (R$)"
coluna_producao = "Produção (t)"
credito_municipal = (
    credito_g3.merge(anos_g3, on=chave_ano_g3)
    .groupby(chave_municipio_g3)[coluna_credito].sum()
)
producao_municipal = (
    producao_g3.merge(anos_g3, on=chave_ano_g3)
    .groupby(chave_municipio_g3)[coluna_producao].sum()
)
# Incluir municípios com dados em uma só fonte, usando zero no outro indicador.
totais_g3 = pd.concat([credito_municipal, producao_municipal], axis=1).fillna(0).reset_index()


def grafico_3(cultura):
    dados = totais_g3[totais_g3["cultura"] == cultura]
    faixas = [1, 5, 10]
    rotulos = [f"Top {percentual}%" for percentual in faixas] + ["Demais municípios"]
    resumo = pd.DataFrame(index=rotulos)

    for indicador in [coluna_credito, coluna_producao]:
        valores = dados[indicador].sort_values(ascending=False)
        total = valores.sum()
        participacoes = []
        for percentual in faixas:
            # Arredondar para cima para incluir pelo menos um município.
            quantidade = (len(valores) * percentual + 99) // 100
            parcela = valores.head(quantidade).sum()
            participacoes.append(parcela / total if total > 0 else float("nan"))
        # Somar o que ficou fora do Top 10% no ranking deste indicador.
        restante = valores.iloc[quantidade:].sum()
        participacoes.append(restante / total if total > 0 else float("nan"))
        resumo[indicador] = participacoes

    primeiro_ano, ultimo_ano = periodos_g3.loc[cultura]
    ax = (resumo * 100).plot.bar(
        ylabel="Participação no total nacional (%)",
        color=["#2866A6", "#3A851D"],
        figsize=(11, 6),
        rot=0,
        width=0.7,
        ylim=(0, 100),
    )
    ax.set_title(
        f"Concentração por município · {cultura} ({primeiro_ano} a {ultimo_ano})", pad=32,
    )
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
    ax.get_figure().tight_layout()
    plt.show()
    plt.close(ax.get_figure())

seletor_g3 = Dropdown(
    options=periodos_g3.index.tolist(), description="Cultura", layout=Layout(width="330px"),
)
saida_g3 = interactive_output(grafico_3, {"cultura": seletor_g3})
display(seletor_g3, saida_g3)

Dropdown(description='Cultura', layout=Layout(width='330px'), options=('Cana-de-açúcar', 'Milho (em grão)', 'S…

Output()